In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split,cross_val_score,cross_val_predict,StratifiedKFold
from sklearn.metrics import accuracy_score ,precision_score ,recall_score,f1_score
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

In [2]:
(datetime.now()+timedelta(days=1)).strftime("%Y-%m-%d")

'2025-11-29'

In [3]:
# ----------------------------------------------------------
# CONFIG
# ----------------------------------------------------------
END_DATE = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
ROLL_7 = 7
ROLL_14 = 14
ROLL_30 = 30


# ----------------------------------------------------------
# FUNCTION: Fetch New Price Data
# ----------------------------------------------------------
def fetch_price_data(asset_symbol, start_date, end_date):
    data = yf.download(asset_symbol, start=start_date, end=end_date)
    data = data.reset_index()
    data = data[["Date", "Close"]]
    data.columns = ["date", "final_price"]
    data["asset"] = asset_symbol
    return data


# ----------------------------------------------------------
# FUNCTION: Feature Engineering
# ----------------------------------------------------------
def add_features(df):

    df = df.sort_values(["asset", "date"]).reset_index(drop=True)
    g = df.groupby("asset")

    df["7d_avg"] = g["final_price"].transform(lambda x: x.rolling(ROLL_7).mean())
    df["30d_avg"] = g["final_price"].transform(lambda x: x.rolling(ROLL_30).mean())
    df["daily_pct_change"] = g["final_price"].transform(lambda x: x.pct_change())
    df["volatility_7d"] = g["final_price"].transform(lambda x: x.rolling(ROLL_7).std())
    df["momentum_14d"] = g["final_price"].transform(lambda x: x - x.shift(ROLL_14))

    df["price_zscore"] = g["final_price"].transform(
        lambda x: (x - x.rolling(ROLL_30).mean()) / x.rolling(ROLL_30).std()
    )

    df["trend_signal"] = g.apply(
        lambda x: (x["7d_avg"] > x["30d_avg"]).astype(int).shift(-1)
    ).reset_index(level=0, drop=True)

    df = df.dropna()
    return df


# ----------------------------------------------------------
# CLEAN NAME MAP
# ----------------------------------------------------------
symbol_map = {
    "BTC-USD": "bitcoin",
    "ETH-USD": "ethereum",
    "LTC-USD": "litecoin",
    "XRP-USD": "ripple",
    "GC=F": "gold",
    "SI=F": "silver"
}

reverse_map = {v: k for k, v in symbol_map.items()}


# ----------------------------------------------------------
# FUNCTION: Remove old rows
# ----------------------------------------------------------
def remove_old_rows(full_df, asset_symbol, num_rows):
    asset_rows = full_df[full_df["asset"] == asset_symbol]
    delete_rows = asset_rows.head(num_rows)
    return full_df.drop(delete_rows.index)


# ----------------------------------------------------------
# MAIN PIPELINE
# ----------------------------------------------------------
def build_dataset(old_df, asset_symbol):

    old_df = old_df.copy()

    # Make sure assets are original symbols always
    old_df["asset"] = old_df["asset"].replace(reverse_map)

    old_df["date"] = pd.to_datetime(old_df["date"])
    old_df["final_price"] = old_df["final_price"].astype(str).str.replace("$", "").astype(float)

    # Find last available date
    last_date = old_df[old_df["asset"] == asset_symbol]["date"].max()

    if pd.isna(last_date):
        start_date = "2010-01-01"
    else:
        start_date = (last_date + timedelta(days=1)).strftime("%Y-%m-%d")

    # Fetch new rows
    new_df = fetch_price_data(asset_symbol, start_date, END_DATE)

    # Merge
    full_df = pd.concat([old_df, new_df], ignore_index=True)
    full_df["date"] = pd.to_datetime(full_df["date"])

    # Number of new rows fetched for this asset
    new_rows_count = new_df.shape[0]

    # Remove equal old rows BEFORE feature engineering
    full_df = remove_old_rows(full_df, asset_symbol, new_rows_count)

    # Now add features
    full_df = add_features(full_df)

    return full_df


# ----------------------------------------------------------
# RUN PIPELINE
# ----------------------------------------------------------
old_df = pd.read_csv("/content/multi_asset_market_data.csv")

# Ensure assets are original at start
old_df["asset"] = old_df["asset"].replace(reverse_map)

assets_list = list(symbol_map.keys())

final_dataset = old_df.copy()

for asset in assets_list:
    final_dataset = build_dataset(final_dataset, asset)

# Convert to human names ONLY at the end
final_dataset["asset"] = final_dataset["asset"].replace(symbol_map)

print("\n=== FINAL DATASET SHAPE ===")
print(final_dataset.shape)



/tmp/ipython-input-1656677734.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset_symbol, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-1656677734.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df["trend_signal"] = g.apply(
/tmp/ipython-input-1656677734.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset_symbol, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-1656677734.py:40: DeprecationWarning: DataFrameGroupBy.apply operated o


=== FINAL DATASET SHAPE ===
(48920, 10)


/tmp/ipython-input-1656677734.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df["trend_signal"] = g.apply(


In [4]:
final_dataset["date"].max()

Timestamp('2025-11-25 00:00:00')

In [5]:
final_dataset.shape

(48920, 10)

In [6]:
df=final_dataset

In [7]:
df=final_dataset.drop(columns=["date","final_price"])

In [8]:
df=pd.get_dummies(df,columns=["asset"],dtype=int)

In [9]:
X=df.drop('trend_signal',axis=1)
Y=df[['trend_signal']]

In [10]:
Training_columns=list(X.columns)

In [11]:
# X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=40)

In [12]:
xgb_model = XGBClassifier(
    n_estimators=400,           # Number of boosting rounds
    learning_rate=0.05,         # Smaller = slower but better accuracy
    max_depth=8,                # Depth of individual trees
    subsample=0.8,              # Randomly sample 80% of training data per round
    colsample_bytree=0.8,       # Randomly sample 80% of features
    gamma=1,                    # Minimum loss reduction to make a split
    reg_lambda=1.5,             # L2 regularization
    reg_alpha=0.5,              # L1 regularization
    min_child_weight=5,         # Minimum instance weight per child
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

In [13]:
xgb_model.fit(X,Y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=1,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=5, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=-1,
              num_parallel_tree=None, ...)

In [14]:
# y_pred=xgb_model.predict(X_test)

In [15]:
# accuracy_score(Y_test,y_pred)

In [16]:

# print("Train Accuracy:", xgb_model.score(X_train, Y_train))
# print("Test Accuracy:", xgb_model.score(X_test, Y_test))

# y_pred = xgb_model.predict(X_test)

# print("\nClassification Report:\n")
# print(classification_report(Y_test, y_pred))

# print("\nConfusion Matrix:\n")
# print(confusion_matrix(Y_test, y_pred))


# WEBSITE

In [17]:
from ipywidgets import Dropdown
from IPython.display import display


In [18]:
asset_dropdown = Dropdown(
    options=["bitcoin", "ethereum", "litecoin", "ripple", "gold", "silver"],
    value="bitcoin",       # default selected value
    description="Asset:"
)

display(asset_dropdown)


Dropdown(description='Asset:', options=('bitcoin', 'ethereum', 'litecoin', 'ripple', 'gold', 'silver'), value=…

In [49]:
asset_dropdown.value

'gold'

In [50]:
today = datetime.now()+timedelta(days=1)
today

datetime.datetime(2025, 11, 29, 18, 13, 57, 256574)

In [51]:
#///////////////////   Lines of code for backedn  *******************************


# Extracting date to fetching data from yfinanace
from datetime import datetime, timedelta

# Aaj ki date
today = datetime.now()

# 60 din peechay (Minus)
sixty_days_ago = today - timedelta(days=90)
one_days_after = today + timedelta(days=1)
# Format convert karna (YYYY-MM-DD)
start_date = sixty_days_ago.strftime("%Y-%m-%d")
one_day = one_days_after.strftime("%Y-%m-%d")




assets_list = [
    "BTC-USD",
    "ETH-USD",
    "LTC-USD",
    "XRP-USD",
    "GC=F",
    "SI=F"
]




# ***************************  Lines of code for fetching data and adding Features *******************************************


# ----------------------------------------------------------
# FUNCTION: Fetch Price Data from yfinance
# ----------------------------------------------------------
def fetch_price_data(asset_symbol, start_date, end_date):
    data = yf.download(asset_symbol, start=start_date, end=end_date)
    data = data.reset_index()
    data = data[["Date", "Close"]]
    data.columns = ["date", "final_price"]
    data["asset"] = asset_symbol
    return data



# ----------------------------------------------------------
# FUNCTION: Feature Engineering
# ----------------------------------------------------------
def add_features(df):

    # Always group by asset
    df = df.sort_values(["asset", "date"]).reset_index(drop=True)

    g = df.groupby("asset")

    df["7d_avg"] = g["final_price"].transform(lambda x: x.rolling(7).mean())
    df["30d_avg"] = g["final_price"].transform(lambda x: x.rolling(30).mean())

    df["daily_pct_change"] = g["final_price"].transform(lambda x: x.pct_change())

    df["volatility_7d"] = g["final_price"].transform(lambda x: x.rolling(7).std())

    df["momentum_14d"] = g["final_price"].transform(lambda x: x - x.shift(14))

    df["price_zscore"] = g["final_price"].transform(
        lambda x: (x - x.rolling(30).mean()) / x.rolling(30).std()
    )

    return df



main_df = []

for i in assets_list:
    df = fetch_price_data(i, start_date, one_day)
    df = add_features(df)
    main_df.append(df.tail(1))

main_df = pd.concat(main_df, ignore_index=True)


/tmp/ipython-input-1951245428.py:39: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset_symbol, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-1951245428.py:39: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset_symbol, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-1951245428.py:39: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset_symbol, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-1951245428.py:39: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset_symbol, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 comple

In [52]:
symbol_to_name = {
    "BTC-USD": "bitcoin",
    "ETH-USD": "ethereum",
    "LTC-USD": "litecoin",
    "XRP-USD": "ripple",
    "GC=F": "gold",
    "SI=F": "silver"
}
main_df["asset"] = main_df["asset"].replace(symbol_to_name)
main_df

,date,final_price,asset,7d_avg,30d_avg,daily_pct_change,volatility_7d,momentum_14d,price_zscore
0,2025-11-28,90709.500000,bitcoin,88511.294643,97485.327865,-0.006309,2443.191566,-3688.289062,-0.823178
1,2025-11-28,3036.428955,ethereum,2936.959473,3260.231877,0.007260,109.420738,-67.356689,-0.662469
2,2025-11-28,83.412041,litecoin,84.688786,93.105639,-0.037115,1.842590,-14.102089,-1.242474
3,2025-11-28,2.163790,ripple,2.144191,2.260073,-0.016827,0.105696,-0.080192,-0.607565
4,2025-11-28,4251.100098,gold,4097.642857,4086.109961,0.038906,68.669457,270.800049,1.665491
5,2025-11-28,57.084999,silver,51.337857,49.684233,0.135004,2.551129,9.222000,3.178842


In [53]:
row=main_df[main_df["asset"]==asset_dropdown.value]

In [54]:
row

,date,final_price,asset,7d_avg,30d_avg,daily_pct_change,volatility_7d,momentum_14d,price_zscore
4,2025-11-28,4251.100098,gold,4097.642857,4086.109961,0.038906,68.669457,270.800049,1.665491


In [55]:
last=pd.get_dummies(row,columns=["asset"],dtype=int)

In [56]:
last

,date,final_price,7d_avg,30d_avg,daily_pct_change,volatility_7d,momentum_14d,price_zscore,asset_gold
4,2025-11-28,4251.100098,4097.642857,4086.109961,0.038906,68.669457,270.800049,1.665491,1


In [57]:
last.drop(columns=["date","final_price"],inplace=True)

In [58]:
last.columns

Index(['7d_avg', '30d_avg', 'daily_pct_change', 'volatility_7d',
       'momentum_14d', 'price_zscore', 'asset_gold'],
      dtype='object')

In [59]:
Training_columns

['7d_avg',
 '30d_avg',
 'daily_pct_change',
 'volatility_7d',
 'momentum_14d',
 'price_zscore',
 'asset_bitcoin',
 'asset_ethereum',
 'asset_gold',
 'asset_litecoin',
 'asset_ripple',
 'asset_silver']

In [60]:
new_last_row=last.reindex(columns=Training_columns,fill_value=0.0)

In [61]:
new_last_row

,7d_avg,30d_avg,daily_pct_change,volatility_7d,momentum_14d,price_zscore,asset_bitcoin,asset_ethereum,asset_gold,asset_litecoin,asset_ripple,asset_silver
4,4097.642857,4086.109961,0.038906,68.669457,270.800049,1.665491,0.0,0.0,1,0.0,0.0,0.0


In [62]:
trend=xgb_model.predict(new_last_row)
trend

array([1])

In [63]:
if(trend==1):
  print(f"Trendig of {asset_dropdown.value} in Future will upward" )
else:
   print(f"Trendig of {asset_dropdown.value} in future will downward" )

Trendig of gold in Future will upward
